# SOM

En este cuaderno se va a investigar si las señales del bosson de Higgs y las señales de fondo se agrupan entre ellas de forma natural o por el contrario se mezclan. 

Para esta parte de análisis se van a utilizar los conjuntos de entrenamiento, tanto el que tiene solo las variables PRI como el que tiene ambas (PRI y DER).

In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [38]:
Entrenamiento = pd.read_csv("Entrenamiento.csv")

## SOM para csv Entrenamiento_PRI

In [39]:
Entrenamiento_DER = Entrenamiento.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight"])
Entrenamiento_PRI = Entrenamiento_DER[["PRI_tau_pt", "PRI_tau_eta", "PRI_tau_phi","PRI_lep_pt","PRI_lep_eta","PRI_lep_phi","PRI_met","PRI_met_phi","PRI_met_sumet","PRI_jet_num","PRI_jet_leading_pt","PRI_jet_leading_eta","PRI_jet_leading_phi","PRI_jet_subleading_pt", "PRI_jet_subleading_eta","PRI_jet_subleading_phi","PRI_jet_all_pt"]].copy().reset_index(drop=True)

Entrenamiento_PRI.shape

(250000, 17)

### Configuración y justificación de los hiperparámetros del SOM

Se han elegido 4 épocas, es decir, 1000000 iteraciones, porque son suficientes para que el mapa se organice y podamos sacar conclusiones.  

La heurística de Vesanto (5√N) daría unas 2500 neuronas, es decir, una rejilla de dimensión 50×50 neuronas, con unos 2500 eventos por neurona de media, lo que permite que varios eventos compartan neurona y se formen grupos. Sin embargo, se va a implementar una más pequeña de 20x20 para visualizar de forma más clara las agrupaciones que se han descrito al principio. Por eso, la matriz de pesos es de 20x20: 400 neuronas con 17 pesos cada una.  

Como función de vecindad usamos la gaussiana, porque la influencia de la BMU disminuye de forma suave con la distancia en la rejilla: las neuronas cercanas se actualizan más que las lejanas, lo que ayuda a que las neuronas vecinas acaben representando datos parecidos.  
El radio inicial será de σ₀ = 10, que es la mitad del lado, y bajará hasta 1. La tasa de aprendizaje baja de 0,1 a 0,01, unos valores de partida habituales. En ambos casos, calculamos $\tau$ para que lleguen a su valor final en la última iteración.  

In [40]:
# número de iteraciones 
EPOCAS = 4
MAX_ITERACIONES = EPOCAS * len(Entrenamiento_PRI)

rng = np.random.default_rng(42)
entrenamiento_PRI_numpy = Entrenamiento_PRI.to_numpy()
pesos = rng.random((400, 17)) # el vector de cada neurona
neuronas = np.indices((20,20)).reshape(2, -1).T # posición de cada neurona en la matriz final

# distancia euclídea
def distancia(posicion1, posicion2):
    return np.sqrt(np.sum((posicion1 - posicion2)**2, axis=-1))

# radio de vecindad
RADIO_INICIAL = 10 # queremos que baje a radio = 1
RADIO_FINAL = 1
TAU_RADIO = MAX_ITERACIONES / np.log(RADIO_INICIAL/RADIO_FINAL) # tau = T / (ln(RADIO_INICIAL / RADIO_FINAL))

def radio_vecindad(t):
    return RADIO_INICIAL * np.exp(-t / TAU_RADIO)

# función de vecindad
def vecindad(t, posicion_bmu, posicion_neurona):
    return np.exp(-(distancia(posicion_bmu, posicion_neurona))**2/ (2*radio_vecindad(t)**2))

# tasa de aprendizaje
TASA_INICIAL = 0.1 # queremos que baje a 0.01
TASA_FINAL = 0.01
TAU_TASA = MAX_ITERACIONES / np.log(TASA_INICIAL / TASA_FINAL)

def tasa_aprendizaje(t):
    return TASA_INICIAL * np.exp(-t/TAU_TASA)

### Entrenamiento

In [41]:
# función BMU: elige la neurona más parecida

def bmu(vector_entrada, vector_pesos):
    return np.argmin(distancia(vector_entrada, vector_pesos)) 

def adaptacion_pesos(vector_pesos, tasa_aprendizaje, f_vecindad, vector_entrada):
    return vector_pesos + tasa_aprendizaje * f_vecindad * (vector_entrada - vector_pesos)

t = 1
for i in range(EPOCAS):
    orden = rng.permutation(len(entrenamiento_PRI_numpy))  # barajar en cada época
    for j in orden:
        x = entrenamiento_PRI_numpy[j]
        c = bmu(x, pesos)
        t_aprendizaje = tasa_aprendizaje(t)


        f_vecindad = vecindad(t, neuronas[c], neuronas)   # 400 valores de golpe
        pesos = adaptacion_pesos(pesos, t_aprendizaje, f_vecindad[:, None], x)
        t = t + 1

### Resultados, visualizaciones y análisis

Se va a visualizar si estos dos tipos se diferencian de forma natural o no. Para ello se va a identificar el grupo dominante de cada neurona.

In [ ]:
LADO = 20
distancias_mapa = np.zeros((LADO, LADO))
for k in range(len(pesos)):
    vecinas = distancia(neuronas[k], neuronas) == 1
    fila, columna = neuronas[k]
    distancias_mapa[fila, columna] = distancia(pesos[k], pesos[vecinas]).mean()

n_total = np.bincount(bmus, minlength=len(pesos))
n_senal = np.bincount(bmus, weights=es_senal, minlength=len(pesos))

fraccion = np.full(len(pesos), np.nan)          # NaN = neurona sin eventos
ocupadas = n_total > 0

fraccion[ocupadas] = n_senal[ocupadas] / n_total[ocupadas]
mapa_senal = np.full((LADO, LADO), np.nan)
mapa_senal[neuronas[:, 0], neuronas[:, 1]] = fraccion

fig, ax = plt.subplots(figsize=(7, 6))
imagen = ax.imshow(mapa_senal, cmap="coolwarm", norm=norma)
umbral_frontera = np.percentile(distancias_mapa, 80)   # el 20 % de distancias más altas
ax.contour(distancias_mapa, levels=[umbral_frontera], colors="black", linewidths=1)
fig.colorbar(imagen, label="Fracción de eventos de señal")
ax.set_title("Fracción de señal y fronteras del mapa")
plt.show()

NameError: name 'pokemon_norm' is not defined